In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
import os
%matplotlib inline

os.makedirs('../results/eda', exist_ok=True)

In [ ]:
metadata_path = '../data/HAM10000_metadata.csv'
df = pd.read_csv(metadata_path) if os.path.exists(metadata_path) else pd.DataFrame(columns=['image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'patient_id'])
print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index)
    plt.title('Class Distribution')
    total = len(df)
    for p in ax.patches:
        count = int(p.get_height())
        pct = f'{100 * count / total:.1f}%'
        ax.annotate(f'{count}\n({pct})', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
    plt.savefig('../results/eda/class_distribution.png')
    plt.show()

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df, x='dx', order=df['dx'].value_counts().index)
    plt.title('Extreme Class Imbalance')
    ax.annotate('67% of data is nv\n(Severe Imbalance)', xy=(0, len(df[df['dx']=='nv'])), xytext=(2, len(df[df['dx']=='nv'])*0.8),
                arrowprops=dict(facecolor='red', shrink=0.05), fontsize=12, color='red')
    plt.savefig('../results/eda/imbalance_highlight.png')
    plt.show()

In [ ]:
if not df.empty:
    severity_map = {'nv': 'Low Risk', 'df': 'Low Risk', 'bkl': 'Low Risk', 
                    'vasc': 'Refer Soon', 'akiec': 'Refer Soon', 
                    'bcc': 'Refer Immediately', 'mel': 'Refer Immediately'}
    df['severity'] = df['dx'].map(severity_map)
    counts = df['severity'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['#2ca02c', '#d62728', '#ff7f0e'])
    plt.title('Severity Distribution')
    plt.savefig('../results/eda/severity_pie.png')
    plt.show()

In [ ]:
if not df.empty and os.path.exists('../data/images'):
    fig, axes = plt.subplots(7, 3, figsize=(10, 20))
    for i, dx in enumerate(df['dx'].unique()):
        sample_imgs = df[df['dx'] == dx].sample(3, replace=True)['image_id'].values
        for j, img_id in enumerate(sample_imgs):
            img_path = f'../data/images/{img_id}.jpg'
            if os.path.exists(img_path):
                img = Image.open(img_path)
                axes[i, j].imshow(img)
            axes[i, j].set_title(dx)
            axes[i, j].axis('off')
    plt.tight_layout()
    plt.savefig('../results/eda/sample_images.png')
    plt.show()

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='dx', y='age')
    plt.title('Age Distribution by Class')
    plt.savefig('../results/eda/age_boxplot.png')
    plt.show()

In [ ]:
if not df.empty:
    loc_df = pd.crosstab(df['dx'], df['localization'])
    plt.figure(figsize=(12, 6))
    sns.heatmap(loc_df, annot=True, fmt='d', cmap='YlGnBu')
    plt.title('Localization Frequency by Class')
    plt.savefig('../results/eda/localization_heatmap.png')
    plt.show()

In [ ]:
if not df.empty:
    patient_counts = df['patient_id'].value_counts()
    plt.figure(figsize=(10, 6))
    sns.histplot(patient_counts, bins=20)
    plt.title('Images per Patient Distribution')
    plt.xlabel('Number of Images')
    plt.savefig('../results/eda/patient_images_hist.png')
    plt.show()

In [ ]:
# Demonstrating the split sizes
if not df.empty:
    import sys
    sys.path.append('../src')
    try:
        from dataset import patient_level_split
        train, val, test = patient_level_split(df)
        split_df = pd.DataFrame({
            'Train': train['dx'].value_counts(),
            'Val': val['dx'].value_counts(),
            'Test': test['dx'].value_counts()
        }).fillna(0)
        print(split_df)
        split_df.plot(kind='bar', stacked=True, figsize=(10, 6))
        plt.title('Data Split Summary per Class')
        plt.savefig('../results/eda/split_summary.png')
        plt.show()
    except ImportError:
        print('dataset module not found.')

## Summary
- **Imbalance**: The dataset is highly imbalanced with 'nv' comprising ~67% of cases. SMOTE or class weights are required.
- **Domain Shift**: High variance in image capturing devices and vignette presence necessitates data augmentation like `remove_vignette`.
- **Multiple Lesions per Patient**: Shown by the Images per Patient histogram. Strict patient-level train/val/test splitting must be used to prevent data leakage.